# Stage 3 Qwen2.5-VL-3B LoRA overfit gate

This run checks whether a minimal LoRA recipe can preserve JSON generation on a tiny supervised subset before any full validation run. The gate is simple: train on five clean crops, generate on those five crops, and save artifacts for inspection. If this gate fails, the LoRA recipe is not promoted.


In [ ]:
import json
import shutil
import subprocess
from pathlib import Path
from collections import Counter

import pandas as pd
from IPython.display import display

REPO_URL = "https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git"
REPO_DIR = Path("/kaggle/working/vlm-for-insulator-defect-detection")
DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"),
    Path("/kaggle/input/idid-coco-v3"),
]
STAGE3_REL = Path("stage3_regrouped_v2")
TRAIN_JSONL_REL = STAGE3_REL / "train_balanced/vlm_labels_v1_train_balanced_v2.annotated.jsonl"
VAL_JSONL_REL = STAGE3_REL / "val/vlm_labels_v1_val_v2.annotated.jsonl"

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
RUN_NAME = "stage3_qwen25vl_3b_lora_overfit_gate_clean"
PROMPT_VERSION = "qwen_vlm_labels_v1_prompt_v7f_flashover_unclear_to_unknown_nocroppath"
OVERFIT_SAMPLES_PER_CLASS = 2
MAX_OVERFIT_SAMPLES = 5
NUM_EPOCHS = 6
LR = 5e-5
MAX_NEW_TOKENS = 220
MAX_PIXELS = 401408
ARCHIVE_PATH = Path("/kaggle/working/stage3_deliverables_qwen25vl_3b_lora_overfit_gate_clean.tar.gz")

print("RUN_NAME:", RUN_NAME)
print("MODEL_ID:", MODEL_ID)


In [ ]:
def sh(args, cwd: Path | None = None, check: bool = True):
    print("$", " ".join(str(a) for a in args))
    p = subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(str(a) for a in args)}")
    return p

def load_jsonl(path: Path):
    rows = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def write_jsonl(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def resolve_crop(row, split_root: Path) -> Path:
    p = Path(row["crop_path"])
    if p.is_absolute() and p.exists():
        return p
    for cand in [split_root / p, split_root.parent / p]:
        if cand.exists():
            return cand
    raise FileNotFoundError(row["crop_path"])


In [ ]:
gpu = sh(["nvidia-smi"], check=False)
if gpu.returncode != 0:
    raise RuntimeError("GPU is required for this run")


In [ ]:
DATA_ROOT = None
for root in DATASET_ROOT_CANDIDATES:
    if (root / TRAIN_JSONL_REL).exists() and (root / VAL_JSONL_REL).exists():
        DATA_ROOT = root
        break
if DATA_ROOT is None:
    raise FileNotFoundError("stage3_regrouped_v2 train/val JSONL was not found in Kaggle inputs")

train_jsonl = DATA_ROOT / TRAIN_JSONL_REL
val_jsonl = DATA_ROOT / VAL_JSONL_REL
train_root = train_jsonl.parent
val_root = val_jsonl.parent
print("DATA_ROOT:", DATA_ROOT)
print("train_jsonl:", train_jsonl)
print("val_jsonl:", val_jsonl)


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
sh(["git", "clone", REPO_URL, str(REPO_DIR)])
sh([
    "python", "-m", "pip", "install", "-q",
    "transformers==4.57.1", "accelerate", "qwen-vl-utils", "peft", "bitsandbytes", "pyyaml", "tabulate",
], cwd=REPO_DIR)
print("Repo ready:", REPO_DIR)


In [ ]:
import re
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info

system_prompt = (REPO_DIR / "configs/pipeline/prompts/stage3_vlm_system_v7f_flashover_unclear_to_unknown.txt").read_text(encoding="utf-8")
user_template = (REPO_DIR / "configs/pipeline/prompts/stage3_vlm_user_v6d_balanced_notaglock_nocroppath.txt").read_text(encoding="utf-8")
if "crop_path" in user_template:
    raise RuntimeError("Prompt text contains crop_path")

ALLOWED = {"insulator_ok", "defect_flashover", "defect_broken"}

def render_user(row):
    text = user_template
    for key in ["record_id", "split", "source", "bbox_xywh"]:
        text = text.replace("{{" + key + "}}", str(row.get(key, "")))
    return text

def target_json(row):
    obj = {
        "coarse_class": row["coarse_class"],
        "visual_evidence_tags": row.get("visual_evidence_tags", []),
        "visibility": row.get("visibility", "clear"),
        "short_canonical_description_en": row.get("short_canonical_description_en") or row.get("short_canonical_description", ""),
        "report_snippet_en": row.get("report_snippet_en") or row.get("report_snippet", ""),
    }
    return json.dumps(obj, ensure_ascii=False)

def select_overfit_rows(rows):
    selected = []
    counts = Counter()
    for row in rows:
        cls = row.get("coarse_class")
        if cls in ALLOWED and counts[cls] < OVERFIT_SAMPLES_PER_CLASS:
            selected.append(row)
            counts[cls] += 1
        if len(selected) >= MAX_OVERFIT_SAMPLES:
            break
    return selected

train_rows_all = [r for r in load_jsonl(train_jsonl) if r.get("coarse_class") in ALLOWED]
val_rows = [r for r in load_jsonl(val_jsonl) if r.get("coarse_class") in ALLOWED]
overfit_rows = select_overfit_rows(train_rows_all)
print("train all:", len(train_rows_all), Counter(r["coarse_class"] for r in train_rows_all))
print("overfit rows:", len(overfit_rows), Counter(r["coarse_class"] for r in overfit_rows))
print("val rows:", len(val_rows), Counter(r["coarse_class"] for r in val_rows))

device_dtype = torch.float16
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, min_pixels=None, max_pixels=MAX_PIXELS)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=device_dtype, bnb_4bit_quant_type="nf4")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

lora_cfg = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


In [ ]:
class VLDataset(Dataset):
    def __init__(self, rows, root):
        self.rows = rows
        self.root = root
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        image_path = resolve_crop(row, self.root).resolve().as_uri()
        prompt_messages = [
            {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
            {"role": "user", "content": [{"type": "image", "image": image_path}, {"type": "text", "text": render_user(row)}]},
        ]
        full_messages = prompt_messages + [
            {"role": "assistant", "content": [{"type": "text", "text": target_json(row)}]},
        ]
        return {"row": row, "prompt_messages": prompt_messages, "full_messages": full_messages}

def build_inputs(prompt_messages, full_messages):
    prompt_text = processor.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    full_text = processor.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)
    prompt_imgs, prompt_vids = process_vision_info(prompt_messages)
    full_imgs, full_vids = process_vision_info(full_messages)

    prompt_inputs = processor(text=[prompt_text], images=prompt_imgs, videos=prompt_vids, padding=False, return_tensors="pt")
    full_inputs = processor(text=[full_text], images=full_imgs, videos=full_vids, padding=False, return_tensors="pt")

    labels = full_inputs["input_ids"].clone()
    prompt_len = min(prompt_inputs["input_ids"].shape[1], labels.shape[1] - 1)
    labels[:, :prompt_len] = -100
    if processor.tokenizer.pad_token_id is not None:
        labels[labels == processor.tokenizer.pad_token_id] = -100
    if int((labels != -100).sum().item()) < 8:
        raise RuntimeError("Too few supervised answer tokens after masking")
    full_inputs["labels"] = labels
    return full_inputs

def collate(batch):
    item = batch[0]
    return build_inputs(item["prompt_messages"], item["full_messages"])

loader = DataLoader(VLDataset(overfit_rows, train_root), batch_size=1, shuffle=True, collate_fn=collate)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.0)
model.train()
losses = []
for epoch in range(NUM_EPOCHS):
    epoch_losses = []
    for batch in loader:
        batch = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in batch.items()}
        out = model(**batch)
        loss = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.3)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        value = float(loss.detach().cpu())
        losses.append(value)
        epoch_losses.append(value)
    print("epoch", epoch + 1, "mean_loss", sum(epoch_losses) / len(epoch_losses))
print("loss_first", losses[0], "loss_last", losses[-1])


In [ ]:
out_root = REPO_DIR / "outputs/stage3_vlm_baseline_runs" / RUN_NAME
out_root.mkdir(parents=True, exist_ok=True)
adapter_dir = REPO_DIR / "outputs" / RUN_NAME / "adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(adapter_dir)
processor.save_pretrained(adapter_dir)
print("adapter_dir:", adapter_dir)


In [ ]:
def extract_json(text):
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        raise ValueError("no JSON object in generated text")
    return json.loads(m.group(0))

def fallback_prediction():
    return {
        "coarse_class": "unknown",
        "visual_evidence_tags": ["ambiguous_evidence"],
        "visibility": "ambiguous",
        "short_canonical_description_en": "Uncertain crop.",
        "report_snippet_en": "The crop should be reviewed.",
    }

def generate_for_rows(rows, root, output_prefix=None):
    local_raw, local_parsed, local_pred, local_sample = [], [], [], []
    model.eval()
    model.config.use_cache = True
    with torch.no_grad():
        for idx, row in enumerate(rows, start=1):
            image_path = resolve_crop(row, root).resolve().as_uri()
            messages = [
                {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
                {"role": "user", "content": [{"type": "image", "image": image_path}, {"type": "text", "text": render_user(row)}]},
            ]
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            imgs, vids = process_vision_info(messages)
            inputs = processor(text=[text], images=imgs, videos=vids, padding=True, return_tensors="pt").to(model.device)
            gen = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                repetition_penalty=1.05,
                pad_token_id=processor.tokenizer.eos_token_id,
            )
            gen_trim = gen[:, inputs.input_ids.shape[1]:]
            raw_text = processor.batch_decode(gen_trim, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
            status = "ok"
            errors = []
            try:
                norm = extract_json(raw_text)
            except Exception as exc:
                status = "parse_error"
                errors.append(str(exc))
                norm = fallback_prediction()
            base = {k: row.get(k) for k in ["record_id", "image_id", "box_id", "source", "split", "bbox_xywh", "crop_path", "image_path", "score", "category_name", "label_version"]}
            pred = dict(base)
            pred.update(norm)
            pred["needs_review"] = norm.get("visibility") == "ambiguous"
            pred["short_canonical_description"] = norm.get("short_canonical_description_en", "")
            pred["report_snippet"] = norm.get("report_snippet_en", "")
            local_sample.append({"record_id": row["record_id"], "status": status, "backend_mode": "qwen_hf_lora_overfit_gate", "model_id": MODEL_ID})
            local_raw.append({"record_id": row["record_id"], "raw_text": raw_text})
            local_parsed.append({"record_id": row["record_id"], "status": status, "normalization_errors": errors, "normalized_prediction": norm})
            local_pred.append(pred)
            print(idx, row["record_id"], status, raw_text[:120].replace("\n", " "))
    if output_prefix:
        write_jsonl(out_root / f"{output_prefix}_sample_results.jsonl", local_sample)
        write_jsonl(out_root / f"{output_prefix}_raw_responses.jsonl", local_raw)
        write_jsonl(out_root / f"{output_prefix}_parsed_predictions.jsonl", local_parsed)
        write_jsonl(out_root / f"{output_prefix}_predictions_vlm_labels_v1.jsonl", local_pred)
    return local_sample, local_raw, local_parsed, local_pred

overfit_sample, overfit_raw, overfit_parsed, overfit_pred = generate_for_rows(overfit_rows, train_root, output_prefix="overfit_check")
overfit_ok = sum(1 for r in overfit_parsed if r["status"] == "ok")
print("overfit_parse_ok", overfit_ok, "/", len(overfit_rows))

write_jsonl(out_root / "sample_results.jsonl", overfit_sample)
write_jsonl(out_root / "raw_responses.jsonl", overfit_raw)
write_jsonl(out_root / "parsed_predictions.jsonl", overfit_parsed)
write_jsonl(out_root / "predictions_vlm_labels_v1.jsonl", overfit_pred)
summary = {
    "run_name": RUN_NAME,
    "model_id": MODEL_ID,
    "prompt_version": PROMPT_VERSION,
    "train_jsonl": str(train_jsonl),
    "val_jsonl": str(val_jsonl),
    "overfit_samples": len(overfit_rows),
    "overfit_parse_ok": overfit_ok,
    "overfit_gate_passed": overfit_ok == len(overfit_rows),
    "epochs": NUM_EPOCHS,
    "lr": LR,
    "lora_target_modules": ["q_proj", "v_proj"],
    "notes": "Full validation is intentionally not run unless the overfit gate is clean.",
}
(out_root / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
display(pd.DataFrame([summary]))


In [ ]:
# Validate generated schema artifacts. This is expected to pass only if the overfit gate emitted parseable JSON.
sh(["python", "scripts/validate_vlm_labels_v1.py", "--input", str(out_root / "predictions_vlm_labels_v1.jsonl")], cwd=REPO_DIR, check=False)

package_root = REPO_DIR / "outputs" / "stage3_qwen_lora_overfit_gate_package"
if package_root.exists():
    shutil.rmtree(package_root)
package_root.mkdir(parents=True, exist_ok=True)
shutil.copytree(out_root, package_root / RUN_NAME)
if ARCHIVE_PATH.exists():
    ARCHIVE_PATH.unlink()
sh(["tar", "-czf", str(ARCHIVE_PATH), "-C", str(package_root.parent), package_root.name], check=True)
print("Archive:", ARCHIVE_PATH)
shutil.rmtree(REPO_DIR, ignore_errors=True)
print("Cleaned work repo:", REPO_DIR)
